# MNIST Convolutional Neural Network (Keras)

Handwritten digit classifier built with TensorFlow/Keras — following along with the O'Reilly *Deep Learning* book.

**Architecture:**
`28×28×1 input → Conv(64, 3×3) → MaxPool(2×2) → Conv(64, 3×3) → MaxPool(2×2) → Flatten → Dense(128, tanh) → Dropout(0.3) → Dense(128, tanh) → Dropout(0.3) → Dense(10, softmax)`

**Techniques:** TensorFlow/Keras high-level API · Convolutional + max-pooling layers · ReLU + tanh + softmax activations · Dropout regularisation · Sparse categorical cross-entropy loss · Adam optimiser · `validation_split` + `EarlyStopping` to prevent overfitting

**Classes:** Digits 0–9

This notebook is the Keras counterpart to the from-scratch NumPy MNIST notebook (Network 1). The point is to compare a hand-rolled mini-batch SGD on a plain MLP against the high-level Keras API with a small CNN, and to show the standard tools (dropout, validation split, early stopping) that close the train/test gap.

**Result: ~99.2% test accuracy** with a train/test gap of ~0.4%, training stopped automatically by `EarlyStopping` after ~12 epochs.


In [100]:
%pip install tensorflow 


[notice] A new release of pip is available: 24.2 -> 26.0.1
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [108]:
# ── Imports ───────────────────────────────────────────────────────────────
# MNIST dataset loader (ships with Keras)
from keras.datasets import mnist

# TensorFlow / Keras — the deep-learning framework
import tensorflow as tf
from tensorflow.keras.callbacks import EarlyStopping

# Numerical + plotting helpers
import numpy as np
import matplotlib.pyplot as plt

# ── Quick environment check ───────────────────────────────────────────────
# Confirm which TF version we're on and whether a GPU is visible.
# On a Mac without `tensorflow-metal` installed, GPUs will be empty and
# training will run on CPU — which is fine for a small CNN like this one.
print("TF version:", tf.__version__)
print("GPUs:", tf.config.list_physical_devices("GPU"))
print("CPUs:", tf.config.list_physical_devices("CPU"))


TF version: 2.20.0
GPUs: []
CPUs: [PhysicalDevice(name='/physical_device:CPU:0', device_type='CPU')]


---
## Loading the Data

`mnist.load_data()` returns two tuples — one for training, one for testing — already split for us:

- **`x_train`** — shape `(60000, 28, 28)`. 60,000 images, each a 28×28 grid of integer pixel values in `[0, 255]`.
- **`y_train`** — shape `(60000,)`. The correct digit label (0–9) for each training image.
- **`x_test`** / **`y_test`** — same structure but 10,000 images, kept aside so we can evaluate the *trained* network on data it has never seen.

So `x_train[0]` is the first image (a 28×28 array of pixel intensities) and `y_train[0]` is the digit it shows.

```
x_train = [
    [[...28 pixels...], [...28 pixels...], ... 28 rows],  ← image 0
    [[...28 pixels...], [...28 pixels...], ... 28 rows],  ← image 1
    ...
    [[...28 pixels...], [...28 pixels...], ... 28 rows],  ← image 59,999
]
```

This is the same dataset used in the Grokking notebook (Network 1). The only thing that changes between notebooks is *how* we feed the data into the model and *what* the model itself looks like.


In [109]:
(x_train, y_train), (x_test, y_test) = mnist.load_data()

training_image_count, training_image_height, training_image_width = x_train.shape[0], x_train.shape[1], x_train.shape[2]
training_label_count, training_label_dtype = y_train.shape[0], y_train.dtype
print(f"Training Data:")
print(f"Training Images: {training_image_count}")
print(f"Each image is {training_image_height} x {training_image_width} pixels; totaling {training_image_height * training_image_width} pixels")
print(f"Training Labels: {training_label_count}")
print(f"Each label is a digit 0-9 (dtype: {training_label_dtype})")

test_image_count, test_image_height, test_image_width = x_test.shape[0], x_test.shape[1], x_test.shape[2]
test_label_count, test_label_dtype = y_test.shape[0], y_test.dtype
print(f"\nTest Data:")
print(f"Test Images: {test_image_count}")
print(f"Each image is {test_image_height} x {test_image_width} pixels; totaling {test_image_height * test_image_width} pixels")
print(f"Test Labels: {test_label_count}")
print(f"Each label is a digit 0-9 (dtype: {test_label_dtype})")


Training Data:
Training Images: 60000
Each image is 28 x 28 pixels; totaling 784 pixels
Training Labels: 60000
Each label is a digit 0-9 (dtype: uint8)

Test Data:
Test Images: 10000
Each image is 28 x 28 pixels; totaling 784 pixels
Test Labels: 10000
Each label is a digit 0-9 (dtype: uint8)


---
## Network Hyperparameters

A few values that control the size of the network and how long it trains. Putting them in one place makes them easy to tweak later without hunting through the code.

- **`input_size`** — `(28, 28)`. The shape of one input image. Keras' `Input` / `Conv2D` layers take this directly — we don't have to flatten ourselves like we did in the NumPy version.
- **`hidden_size`** — `128`. The number of neurons in each fully-connected hidden layer that comes after the convolutional stack.
- **`output_size`** — `10`. One output neuron per digit class (0–9).
- **`epochs`** — `60`. The maximum number of full passes over the training data. The actual number of epochs run will usually be much lower because `EarlyStopping` will halt training as soon as the validation loss stops improving.

Note: there's no `learning_rate` (`alpha`) here. The Grokking notebook had to hand-pick one for plain SGD, but we're using the **Adam** optimiser, which adapts the per-weight learning rate automatically.


In [111]:
# ── Hyperparameters ───────────────────────────────────────────────────────
input_size  = (training_image_height, training_image_width)  # (28, 28)
hidden_size = 128   # neurons in each fully-connected hidden layer
output_size = 10    # one output per digit class (0–9)
epochs      = 60    # max epochs; EarlyStopping will usually halt much earlier

---
## Building and Training the Network

Unlike the Grokking notebook — where we wrote forward propagation, backpropagation and the weight updates by hand — here we use TensorFlow/Keras' high-level API. Keras builds the weight matrices for us, handles the forward/backward passes, and runs training through `model.fit()`.

### Why this network looks the way it does

1. **Convolutional front-end.** Two `Conv2D(64, 3×3) → MaxPool(2×2)` blocks. Conv layers exploit the fact that images have *local* structure (a digit's stroke is a small connected pattern, not 784 unrelated pixels). Pooling halves the spatial size after each conv block, so the network sees the image at progressively coarser scales while keeping the parameter count modest.
2. **Dense classifier head.** After the conv stack the feature map is flattened and fed through two `Dense(128, tanh)` layers, with `Dropout(0.3)` between them and again before the softmax output. Dropout randomly zeroes 30% of activations during training, which prevents the dense layers from co-adapting and is the main reason the train/test gap stays small.
3. **Adam instead of vanilla SGD.** The Grokking notebook used plain mini-batch gradient descent with a hand-tuned learning rate. Adam adapts the learning rate per weight, so we don't tune `alpha` ourselves.
4. **Sparse categorical cross-entropy** is the natural loss for integer class labels paired with a softmax output — it converges faster than MSE for classification.

### Why we hold out a validation set

We pass `validation_split=0.1` to `fit()`, which tells Keras to set aside the **last 10% of the training data (6,000 images)** as a held-out validation set. After every epoch, Keras measures `val_loss` and `val_accuracy` on those images — data the model is *not* training on. That gives us an honest read on generalisation in real time, without touching the test set.

### Why we use EarlyStopping

`EarlyStopping(monitor='val_loss', patience=5, restore_best_weights=True)` watches `val_loss` each epoch. If it doesn't improve for 5 epochs in a row, training stops automatically and the model's weights are rolled back to whichever epoch had the best `val_loss`. This means:

- We don't waste compute training past the point of diminishing returns.
- We don't overfit by accident, because the moment the model starts memorising the training set (val loss going up while train loss keeps dropping), we bail out.

The combination of dropout + validation split + early stopping is what closed the train/test gap from ~6% (the original MLP) down to <1% on the CNN.


In [113]:
# ── Training-run settings ─────────────────────────────────────────────────
num_train  = 60000   # use the full MNIST training set
num_test   = 10000   # use the full MNIST test set
batch_size = 100     # mini-batch size — matches Network 1 for a fair comparison

# ── EarlyStopping callback ────────────────────────────────────────────────
# Watch val_loss each epoch. If it doesn't improve for 5 epochs in a row,
# stop training and roll back to whichever epoch had the best val_loss.
callbacks = [EarlyStopping(monitor='val_loss', patience=5, restore_best_weights=True)]

# ── Prepare the data ──────────────────────────────────────────────────────
# 1) Normalise pixel values from 0–255 to 0–1 (helps gradients stay well-scaled)
# 2) Reshape to (N, 28, 28, 1) — Conv2D expects an explicit channel dimension,
#    even when there's only one channel (greyscale).
training_images = (x_train[0:num_train] / 255.0).reshape(num_train, training_image_height, training_image_width, 1)
training_labels = y_train[0:num_train]

test_images     = (x_test[0:num_test]   / 255.0).reshape(num_test,  test_image_height,     test_image_width,     1)
test_labels     = y_test[0:num_test]


# ── Build the model ───────────────────────────────────────────────────────
model = tf.keras.models.Sequential([

    # ── Convolutional feature extractor ──
    # Two Conv→Pool blocks. Each Conv2D learns 64 small (3×3) filters; each
    # MaxPooling2D halves the spatial dimensions.
    # Spatial sizes: 28×28 → 26×26 → 13×13 → 11×11 → 5×5
    tf.keras.layers.Conv2D(64, (3, 3), activation='relu',input_shape=(training_image_height, training_image_width, 1)),
    tf.keras.layers.MaxPooling2D((2, 2)),
    tf.keras.layers.Conv2D(64, (3, 3), activation='relu'),
    tf.keras.layers.MaxPooling2D((2, 2)),

    # ── Deep Neural Network ──
    # Layer_0 flatten the 5×5×64 feature map into a 1,600-length vector.
    tf.keras.layers.Flatten(),

    # Layer_1 first fully-connected hidden layer (128 neurons, tanh activation) with dropout.
    tf.keras.layers.Dense(hidden_size, activation=tf.nn.tanh),
    tf.keras.layers.Dropout(0.3),

    # Layer_2 second fully-connected hidden layer with dropout.
    tf.keras.layers.Dense(hidden_size, activation=tf.nn.tanh),
    tf.keras.layers.Dropout(0.3),

    # Output layer: 10 neurons (one per digit) with softmax,
    tf.keras.layers.Dense(output_size, activation=tf.nn.softmax),
])

# ── Compile ───────────────────────────────────────────────────────────────
# - optimizer: Adam — adaptive per-weight learning rate, no manual tuning needed
# - loss: sparse_categorical_crossentropy — pairs naturally with softmax outputs
#         and uses integer labels (0–9) directly, no one-hot encoding required
# - metrics: accuracy — printed alongside loss each epoch
model.compile(optimizer="adam",
              loss="sparse_categorical_crossentropy",
              metrics=["accuracy"])

# ── Train ─────────────────────────────────────────────────────────────────
# - validation_split=0.1: hold out the last 10% of training data as a
#   validation set so we can monitor generalisation each epoch.
# - callbacks=[EarlyStopping(...)]: stop early once val_loss plateaus.
# - verbose=2: print one clean line per epoch (no live progress bar).
print("Training…\n")
history = model.fit(training_images, 
                    training_labels,
                    batch_size=batch_size,
                    epochs=epochs,
                    validation_split=0.1,
                    callbacks=callbacks,
                    verbose=2)

# ── Evaluate on the held-out test set ─────────────────────────────────────
# This is the only time the test set is touched — it gives us an unbiased
# estimate of how well the model will perform on unseen data.
print("\nEvaluating on test set…")
test_loss, test_accuracy = model.evaluate(test_images, test_labels, verbose=0)

# ── Final summary ─────────────────────────────────────────────────────────
# Pull the last epoch's training/validation numbers from the history object,
# alongside the test-set numbers from model.evaluate(). Showing all three
# (train / val / test) makes it easy to spot overfitting or distribution
# mismatch at a glance.
final_train_accuracy = history.history["accuracy"][-1]
final_train_loss     = history.history["loss"][-1]
final_val_accuracy   = history.history["val_accuracy"][-1]
final_val_loss       = history.history["val_loss"][-1]
epochs_run           = len(history.history["loss"])

print("\n── Results ───────────────────────────────────────────────")
print(f"Epochs run              : {epochs_run} (of {epochs} requested)")
print(f"Final training accuracy : {final_train_accuracy * 100:.1f}%  (loss {final_train_loss:.3f})")
print(f"Final validation acc.   : {final_val_accuracy   * 100:.1f}%  (loss {final_val_loss:.3f})")
print(f"Test accuracy           : {test_accuracy        * 100:.1f}%  (loss {test_loss:.3f})")
print(f"Train/test gap          : {(final_train_accuracy - test_accuracy) * 100:+.1f}%")

Training…

Epoch 1/60


/Users/joehill/Developer/DeepLearning/venv/lib/python3.12/site-packages/keras/src/layers/convolutional/base_conv.py:113: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


540/540 - 6s - 10ms/step - accuracy: 0.9417 - loss: 0.1940 - val_accuracy: 0.9840 - val_loss: 0.0537
Epoch 2/60
540/540 - 5s - 10ms/step - accuracy: 0.9814 - loss: 0.0607 - val_accuracy: 0.9887 - val_loss: 0.0361
Epoch 3/60
540/540 - 5s - 9ms/step - accuracy: 0.9863 - loss: 0.0456 - val_accuracy: 0.9912 - val_loss: 0.0364
Epoch 4/60
540/540 - 5s - 9ms/step - accuracy: 0.9884 - loss: 0.0377 - val_accuracy: 0.9892 - val_loss: 0.0376
Epoch 5/60
540/540 - 5s - 9ms/step - accuracy: 0.9910 - loss: 0.0275 - val_accuracy: 0.9908 - val_loss: 0.0364
Epoch 6/60
540/540 - 5s - 9ms/step - accuracy: 0.9921 - loss: 0.0254 - val_accuracy: 0.9892 - val_loss: 0.0413
Epoch 7/60
540/540 - 5s - 9ms/step - accuracy: 0.9934 - loss: 0.0210 - val_accuracy: 0.9915 - val_loss: 0.0331
Epoch 8/60
540/540 - 5s - 9ms/step - accuracy: 0.9935 - loss: 0.0204 - val_accuracy: 0.9912 - val_loss: 0.0329
Epoch 9/60
540/540 - 5s - 9ms/step - accuracy: 0.9945 - loss: 0.0178 - val_accuracy: 0.9913 - val_loss: 0.0351
Epoch 10/6

In [ ]:
# ── Prediction Inspection: inspect a single prediction ─────────────────────────────
# Run every test image through the network and look at the first one in detail.
classifications = model.predict(test_images, verbose=0)

print(f"\nSample prediction for test image 0:")
print(f"  Network output (softmax probabilities) : {classifications[0]}")
print(f"  Predicted digit (argmax of output)     : {np.argmax(classifications[0])}")
print(f"  True digit (from y_test)               : {test_labels[0]}")

In [ ]:
# ── Layer-by-layer summary of the trained model ──────────────────────────
# Shows each layer's output shape and parameter count, so we can confirm
# the spatial dimensions and where the parameters are concentrated.
model.summary()

---
## Exporting the Model for the Web

For this project the deployment target is the GitHub Pages site, where each network gets its own page with an interactive "draw a digit and let the model guess" widget.

The dense-only network from the Grokking notebook was easy to ship: add the two weight matrices to JSON, and a few lines of hand-written JavaScript can do the matrix multiplications and softmax in the browser. A CNN is more complicated — implementing `Conv2D` and `MaxPooling2D` correctly in JavaScript is a meaningful amount of work, and easy to get subtly wrong.

Instead, we'll use **[TensorFlow.js](https://www.tensorflow.org/js)**, a JavaScript port of TensorFlow that runs directly in the browser. It already knows how to execute every layer type our model uses, so all we have to do is translate our trained Keras model into a format it can load.

The pipeline looks like this:

1. **`model.save("mnist_cnn.keras")`** — write the trained model to a single `.keras` file. This is a zip archive containing the architecture, the learned weights, and the optimiser state.
2. **`tensorflowjs_converter --input_format=keras mnist_cnn.keras tfjs_model/`** — a command-line tool (installed by the `tensorflowjs` Python package below) that reads the `.keras` file and writes out:
   - `model.json` — the architecture and a manifest pointing to the weight shards
   - `group1-shard1of1.bin` — the actual weight tensors as raw binary
3. **In the browser**, the site loads `@tensorflow/tfjs` from a CDN, calls `tf.loadLayersModel("…/model.json")`, and then runs `model.predict()` on whatever the user draws on the canvas.

A couple of notes:

- **`tensorflowjs` is only used for the conversion step.** It's a Python-side tool. The browser doesn't need it — it loads the much smaller `@tensorflow/tfjs` runtime from a CDN.
- **The `.keras` file is throwaway.** Once we have `tfjs_model/`, we don't need the intermediate `.keras` file anymore — it isn't committed to the repo.


### Install the converter

We only need this once. The `tensorflowjs` package brings in the `tensorflowjs_converter` CLI tool that turns a `.keras` file into the browser-friendly `model.json` + binary weight shard format.


In [ ]:
%pip install tensorflowjs

### Save the trained model

`model.save()` writes the *whole* model — architecture, learned weights, and optimiser state — into a single `.keras` zip file. This is the input to the converter.


In [ ]:
# Save the trained model into a single .keras file in this folder.
# This is an intermediate file used by the tensorflowjs_converter step below;
# it isn't committed to the repository.
model.save("mnist_cnn.keras")
print("Saved → mnist_cnn.keras")

### Run the converter

Open a terminal in this folder and run:

```bash
tensorflowjs_converter \
    --input_format=keras \
    mnist_cnn.keras \
    ../docs/networks/2-oreilly-deep-learning-mnist/tfjs_model
```

That writes `model.json` plus a binary weight shard into `docs/networks/2-oreilly-deep-learning-mnist/tfjs_model/`, which is what the GitHub Pages site loads at runtime via `tf.loadLayersModel()`.
